###  1. Defining the schema for the drivers input

In [0]:
%run ../00.Common/01.Environment-config

In [0]:
%run ../00.Common/02.Helper_Notebook

In [0]:
source_file = f"{landing_folder_path}/drivers.json"
table_name = f"{catalog_name}.{bronze_schema}.drivers"

In [0]:
from pyspark.sql.types import StructType, StructField, StringType, IntegerType, DateType
name_schema = StructType([
    StructField('givenName', StringType()),
    StructField('familyName', StringType())
])
driver_schema = StructType([
    StructField('driverId', StringType()),
    StructField('name',name_schema),
    StructField('dateOfBirth', DateType()),
    StructField('url', StringType())
])

### 2. Reading the constructors.json file from the files location

In [0]:
driver_df = (
    spark.read
    .format('json')
    .option('mode','FAILFAST')
    .schema(driver_schema)
    .load(source_file)
     )

### 3. View the cons df

In [0]:
display(driver_df)

### 4. Adding the file ingestion timestamp and sourcefile metadata in the df

In [0]:
final_driver = add_timestamp_metadata(driver_df)

### 5. Writing the final dataframe into the table under bronze schema

In [0]:
(
    final_driver.write.format('delta')
    .mode('overwrite')
    .saveAsTable(table_name)
)

In [0]:
display(spark.table(table_name))